In [9]:
# Setup
import requests
import json
from pathlib import Path
import subprocess
import time

PROJECT_ROOT = Path.cwd().parent.parent
print("✅ Deployment setup complete")

# Test API Locally
def test_api():
    """Test if API is running"""
    try:
        response = requests.get("http://localhost:8000/health", timeout=5)
        if response.status_code == 200:
            print("✅ API is running")
            return True
    except:
        print("❌ API is not running")
        return False

# Check API status
api_running = test_api()

# Start API if not running
if not api_running:
    print("\n🚀 Starting API server...")
    # Start API in background
    subprocess.Popen(["python", str(PROJECT_ROOT / "api.py")], 
                     shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    test_api()

# Test API Endpoints
print("\n📡 Testing API Endpoints:")

# Health check
response = requests.get("http://localhost:8000/health")
print(f"   Health: {response.status_code} - {response.json()['status']}")

# Get prediction
response = requests.get("http://localhost:8000/predict")
if response.status_code == 200:
    data = response.json()
    print(f"   Prediction: {data['prediction_percent']} ({data['direction']})")
    print(f"   Confidence: {data['confidence']}")
    print(f"   Recommendation: {data['recommendation']}")

# Get metrics
response = requests.get("http://localhost:8000/metrics")
if response.status_code == 200:
    data = response.json()
    print(f"   RMSE: {data['rmse']:.4f}")
    print(f"   Model Version: {data['model_version']}")

# Docker Deployment Commands
print("\n🐳 Docker Deployment Commands:")
print("="*50)
print("""
# Build Docker image
docker build -t sp500-predictor:latest .

# Run Docker container
docker run -d -p 8000:8000 --name sp500-api sp500-predictor:latest

# Check logs
docker logs sp500-api -f

# Stop container
docker stop sp500-api

# Remove container
docker rm sp500-api

# Docker Compose (full stack)
docker-compose up -d --build
""")

# Kubernetes Deployment
print("\n☸️ Kubernetes Deployment Commands:")
print("="*50)
print("""
# Apply all manifests
kubectl apply -f kubernetes/

# Check deployment status
kubectl get pods
kubectl get services
kubectl get ingress

# Port forward for local access
kubectl port-forward service/sp500-predictor-api 8000:80

# Scale deployment
kubectl scale deployment sp500-predictor-api --replicas=5

# View logs
kubectl logs -f deployment/sp500-predictor-api

# Update image
kubectl set image deployment/sp500-predictor-api api=sp500-predictor:latest

# Rollback if needed
kubectl rollout undo deployment/sp500-predictor-api

# Delete all resources
kubectl delete -f kubernetes/
""")

# Cloud Deployment Options
print("\n☁️ Cloud Deployment Options:")
print("="*50)
print("""
1. Render.com:
   - Connect GitHub repository
   - Set start command: uvicorn api:app --host 0.0.0.0 --port $PORT
   - Auto-deploy on push

2. Railway.app:
   - railway login
   - railway init
   - railway up

3. Heroku:
   - heroku create sp500-predictor
   - git push heroku main

4. AWS ECS:
   - Push image to ECR
   - Create ECS task definition
   - Deploy to ECS cluster

5. Google Cloud Run:
   - gcloud run deploy sp500-predictor --image=sp500-predictor:latest --port=8000
""")

# Monitoring Setup
print("\n📊 Monitoring Setup:")
print("="*50)
print("""
# Check API health
curl http://localhost:8000/health

# View logs
tail -f logs/api.log

# Check predictions
tail -f logs/predictions.log

# Monitor metrics
watch -n 1 'curl -s http://localhost:8000/metrics | jq .'

# Prometheus metrics
curl http://localhost:8000/metrics
""")

print("\n✅ Deployment notebook ready!")

✅ Deployment setup complete
✅ API is running

📡 Testing API Endpoints:
   Health: 200 - healthy

🐳 Docker Deployment Commands:

# Build Docker image
docker build -t sp500-predictor:latest .

# Run Docker container
docker run -d -p 8000:8000 --name sp500-api sp500-predictor:latest

# Check logs
docker logs sp500-api -f

# Stop container
docker stop sp500-api

# Remove container
docker rm sp500-api

# Docker Compose (full stack)
docker-compose up -d --build


☸️ Kubernetes Deployment Commands:

# Apply all manifests
kubectl apply -f kubernetes/

# Check deployment status
kubectl get pods
kubectl get services
kubectl get ingress

# Port forward for local access
kubectl port-forward service/sp500-predictor-api 8000:80

# Scale deployment
kubectl scale deployment sp500-predictor-api --replicas=5

# View logs
kubectl logs -f deployment/sp500-predictor-api

# Update image
kubectl set image deployment/sp500-predictor-api api=sp500-predictor:latest

# Rollback if needed
kubectl rollout undo dep

In [ ]:
"""
Professional Animated Feature Analysis Dashboard
Save as: scripts/animated_feature_analysis.py
Run: python scripts/animated_feature_analysis.py

Features:
- Interactive correlation matrix heatmap
- Animated feature importance bar chart
- 3D feature correlation network
- Target distribution analysis
- Feature clustering visualization
"""

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Create directories
PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'scripts' else Path.cwd()
Path("visualizations/html").mkdir(parents=True, exist_ok=True)
Path("visualizations/plots").mkdir(parents=True, exist_ok=True)

# Professional color palette
COLORS = {
    'positive': '#2ECC71',
    'negative': '#E74C3C',
    'neutral': '#95A5A6',
    'primary': '#1E88E5',
    'secondary': '#DC143C',
    'warning': '#F39C12',
    'info': '#3498DB',
    'dark': '#2C3E50',
    'light': '#ECF0F1',
    'correlation_high': '#E74C3C',
    'correlation_low': '#3498DB'
}

print("="*60)
print("🎨 Generating Animated Feature Analysis Dashboard")
print("="*60)

# ============================================
# 1. INTERACTIVE CORRELATION MATRIX HEATMAP
# ============================================

def create_interactive_correlation_matrix(df):
    """Create interactive correlation matrix with dendrogram"""
    
    # Calculate correlation matrix
    corr_matrix = df.corr()
    
    # Create hierarchical clustering for better visualization
    dissimilarity = 1 - abs(corr_matrix)
    Z = hierarchy.linkage(squareform(dissimilarity), method='average')
    
    # Get order from dendrogram
    dendro = hierarchy.dendrogram(Z, no_plot=True)
    order = dendro['leaves']
    
    # Reorder matrix
    corr_ordered = corr_matrix.iloc[order, order]
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=corr_ordered.values,
        x=corr_ordered.columns,
        y=corr_ordered.columns,
        colorscale='RdBu',
        zmin=-1,
        zmax=1,
        text=np.round(corr_ordered.values, 2),
        texttemplate='%{text}',
        textfont={"size": 8},
        hovertemplate='<b>%{x}</b> vs <b>%{y}</b><br>' +
                      'Correlation: %{z:.3f}<br>' +
                      '<extra></extra>'
    ))
    
    # Add annotations for high correlations
    high_corr = np.where(np.abs(corr_ordered.values) > 0.8)
    annotations = []
    for i, j in zip(high_corr[0], high_corr[1]):
        if i != j and i < j:  # Avoid duplicates
            annotations.append(dict(
                x=corr_ordered.columns[j],
                y=corr_ordered.columns[i],
                xref="x", yref="y",
                text=f"<b>{corr_ordered.iloc[i, j]:.2f}</b>",
                showarrow=False,
                font=dict(size=9, color='white' if abs(corr_ordered.iloc[i, j]) > 0.9 else 'black')
            ))
    
    fig.update_layout(
        title=dict(
            text='<b>📊 Feature Correlation Matrix</b><br><sub>Hierarchically clustered correlations with dendrogram ordering</sub>',
            x=0.5,
            font=dict(size=16, family='Arial Black')
        ),
        height=900,
        width=1100,
        xaxis=dict(title='Features', tickangle=45, tickfont=dict(size=9)),
        yaxis=dict(title='Features', tickfont=dict(size=9)),
        template='plotly_white'
    )
    
    fig.write_html('visualizations/html/correlation_matrix_interactive.html')
    fig.write_image('visualizations/plots/correlation_matrix_interactive.png', width=1100, height=900)
    
    print("✅ Created: correlation_matrix_interactive.html")
    return fig

# ============================================
# 2. ANIMATED FEATURE IMPORTANCE BAR CHART
# ============================================

def create_animated_feature_importance(df):
    """Create animated feature importance chart with cumulative importance"""
    
    # Prepare data
    X = df.drop('target', axis=1)
    y = df['target']
    
    # Train Random Forest
    print("   Training Random Forest for feature importance...")
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    
    # Create importance dataframe
    importance_df = pd.DataFrame({
        'feature': X.columns,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=True)
    
    # Calculate cumulative importance
    cumulative = np.cumsum(importance_df['importance'].values[::-1])[::-1]
    
    # Create figure with secondary y-axis
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Feature Importance', 'Cumulative Importance'),
        specs=[[{'type': 'bar'}, {'type': 'scatter'}]],
        column_widths=[0.6, 0.4]
    )
    
    # 1. Horizontal bar chart
    fig.add_trace(
        go.Bar(
            x=importance_df['importance'],
            y=importance_df['feature'],
            orientation='h',
            marker=dict(
                color=importance_df['importance'],
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title='Importance', x=1.05)
            ),
            text=importance_df['importance'].apply(lambda x: f'{x:.4f}'),
            textposition='outside',
            hovertemplate='<b>%{y}</b><br>Importance: %{x:.4f}<br>%{text}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # 2. Cumulative importance line
    fig.add_trace(
        go.Scatter(
            x=importance_df['importance'],
            y=importance_df['feature'],
            mode='lines+markers',
            name='Cumulative',
            line=dict(color=COLORS['warning'], width=3),
            marker=dict(size=8, color=COLORS['warning']),
            hovertemplate='<b>%{y}</b><br>Cumulative: %{x:.4f}<extra></extra>',
            xaxis='x2',
            yaxis='y2'
        ),
        row=1, col=2
    )
    
    # Add threshold line at 80%
    cum_values = cumulative[::-1]
    threshold_idx = np.where(cum_values >= 0.8)[0]
    if len(threshold_idx) > 0:
        threshold_feature = importance_df['feature'].iloc[threshold_idx[-1]]
        fig.add_hline(y=threshold_feature, line_dash="dash", 
                     line_color="red", opacity=0.7, row=1, col=2)
        fig.add_annotation(
            x=0.85, y=threshold_feature,
            text="80% Threshold",
            showarrow=True,
            arrowhead=2,
            row=1, col=2
        )
    
    fig.update_layout(
        title=dict(
            text='<b>🏆 Feature Importance Analysis</b><br><sub>Random Forest feature importance with cumulative contribution</sub>',
            x=0.5,
            font=dict(size=16)
        ),
        height=800,
        width=1300,
        template='plotly_white',
        showlegend=False
    )
    
    fig.update_xaxes(title_text='Importance', row=1, col=1)
    fig.update_yaxes(title_text='Features', row=1, col=1)
    fig.update_xaxes(title_text='Cumulative Importance', row=1, col=2, range=[0, 1])
    fig.update_yaxes(title_text='Features', row=1, col=2, matches='y')
    
    fig.write_html('visualizations/html/feature_importance_animated.html')
    fig.write_image('visualizations/plots/feature_importance_animated.png', width=1300, height=800)
    
    print("✅ Created: feature_importance_animated.html")
    return fig, importance_df

# ============================================
# 3. 3D FEATURE SPACE VISUALIZATION (PCA)
# ============================================

def create_3d_feature_space_pca(df):
    """Create 3D PCA visualization of feature space"""
    
    # Prepare data
    X = df.drop('target', axis=1)
    y = df['target']
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Apply PCA
    pca = PCA(n_components=3)
    X_pca = pca.fit_transform(X_scaled)
    
    # Create target categories
    categories = pd.cut(y, bins=[-np.inf, -0.005, 0.005, np.inf], 
                        labels=['Bearish', 'Neutral', 'Bullish'])
    
    # Color mapping
    color_map = {'Bearish': COLORS['negative'], 
                 'Neutral': COLORS['neutral'], 
                 'Bullish': COLORS['positive']}
    
    # Create 3D scatter plot
    fig = go.Figure(data=[go.Scatter3d(
        x=X_pca[:, 0],
        y=X_pca[:, 1],
        z=X_pca[:, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=[color_map[cat] for cat in categories],
            opacity=0.6,
            line=dict(color='white', width=0.5)
        ),
        text=[f'Return: {r:.4f}<br>Direction: {cat}' for r, cat in zip(y, categories)],
        hovertemplate='%{text}<extra></extra>'
    )])
    
    # Add explained variance to labels
    explained_var = pca.explained_variance_ratio_
    
    fig.update_layout(
        title=dict(
            text=f'<b>🎯 3D Feature Space Visualization (PCA)</b><br><sub>PC1: {explained_var[0]:.1%} | PC2: {explained_var[1]:.1%} | PC3: {explained_var[2]:.1%} variance explained</sub>',
            x=0.5,
            font=dict(size=16)
        ),
        scene=dict(
            xaxis_title=f'Principal Component 1 ({explained_var[0]:.1%})',
            yaxis_title=f'Principal Component 2 ({explained_var[1]:.1%})',
            zaxis_title=f'Principal Component 3 ({explained_var[2]:.1%})',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
        ),
        height=700,
        width=1000,
        template='plotly_white'
    )
    
    fig.write_html('visualizations/html/3d_feature_space_pca.html')
    fig.write_image('visualizations/plots/3d_feature_space_pca.png', width=1000, height=700)
    
    print("✅ Created: 3d_feature_space_pca.html")
    return fig

# ============================================
# 4. TARGET DISTRIBUTION ANALYSIS
# ============================================

def create_target_distribution_dashboard(df):
    """Create interactive target distribution dashboard"""
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Target Distribution', 'Box Plot by Direction',
                       'Q-Q Plot', 'Time Series Evolution'),
        specs=[[{'type': 'histogram'}, {'type': 'box'}],
               [{'type': 'scatter'}, {'type': 'scatter'}]]
    )
    
    # 1. Histogram with KDE
    fig.add_trace(
        go.Histogram(
            x=df['target'],
            nbinsx=50,
            name='Returns',
            marker_color=COLORS['primary'],
            opacity=0.7,
            histnorm='probability density',
            hovertemplate='Return: %{x:.4f}<br>Density: %{y:.3f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Add KDE line
    from scipy import stats
    kde = stats.gaussian_kde(df['target'].dropna())
    x_range = np.linspace(df['target'].min(), df['target'].max(), 100)
    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=kde(x_range),
            mode='lines',
            name='KDE',
            line=dict(color=COLORS['warning'], width=2),
            hovertemplate='Return: %{x:.4f}<br>Density: %{y:.3f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Add mean and zero lines
    mean_val = df['target'].mean()
    fig.add_vline(x=mean_val, line_dash="dash", line_color="red", 
                  annotation_text=f'Mean: {mean_val:.4f}', row=1, col=1)
    fig.add_vline(x=0, line_dash="solid", line_color="green", 
                  annotation_text='Zero', row=1, col=1)
    
    # 2. Box plot by direction
    df['direction'] = df['target'] > 0
    bullish_returns = df[df['direction']]['target']
    bearish_returns = df[~df['direction']]['target']
    
    fig.add_trace(
        go.Box(y=bullish_returns, name='Bullish', marker_color=COLORS['positive'],
               boxmean='sd', hovertemplate='Bullish Returns<br>%{y:.4f}<extra></extra>'),
        row=1, col=2
    )
    
    fig.add_trace(
        go.Box(y=bearish_returns, name='Bearish', marker_color=COLORS['negative'],
               boxmean='sd', hovertemplate='Bearish Returns<br>%{y:.4f}<extra></extra>'),
        row=1, col=2
    )
    
    # 3. Q-Q Plot
    sorted_returns = np.sort(df['target'].dropna())
    theoretical_quantiles = stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_returns)))
    
    fig.add_trace(
        go.Scatter(
            x=theoretical_quantiles,
            y=sorted_returns,
            mode='markers',
            marker=dict(size=4, color=COLORS['info'], opacity=0.6),
            name='Q-Q Points',
            hovertemplate='Theoretical: %{x:.3f}<br>Sample: %{y:.4f}<extra></extra>'
        ),
        row=2, col=1
    )
    
    # Reference line
    qq_slope, qq_intercept = np.polyfit(theoretical_quantiles, sorted_returns, 1)
    qq_line = qq_slope * theoretical_quantiles + qq_intercept
    fig.add_trace(
        go.Scatter(
            x=theoretical_quantiles,
            y=qq_line,
            mode='lines',
            name='Reference',
            line=dict(color=COLORS['perfect'], width=2, dash='dash')
        ),
        row=2, col=1
    )
    
    # 4. Time series evolution
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['target'],
            mode='lines',
            name='Returns',
            line=dict(color=COLORS['primary'], width=1.5),
            fill='tozeroy',
            fillcolor='rgba(30, 136, 229, 0.2)',
            hovertemplate='Date: %{x}<br>Return: %{y:.4f}<extra></extra>'
        ),
        row=2, col=2
    )
    
    # Add rolling mean
    rolling_mean = df['target'].rolling(window=20).mean()
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=rolling_mean,
            mode='lines',
            name='20-day MA',
            line=dict(color=COLORS['warning'], width=2, dash='dash'),
            hovertemplate='Date: %{x}<br>MA: %{y:.4f}<extra></extra>'
        ),
        row=2, col=2
    )
    
    # Add zero line
    fig.add_hline(y=0, line_dash="solid", line_color="green", opacity=0.5, row=2, col=2)
    
    # Calculate statistics
    skewness = stats.skew(df['target'].dropna())
    kurtosis = stats.kurtosis(df['target'].dropna())
    
    fig.update_layout(
        title=dict(
            text=f'<b>📈 Target Distribution Analysis</b><br><sub>Skewness: {skewness:.3f} | Kurtosis: {kurtosis:.3f} | Positive Ratio: {(df["target"] > 0).mean():.1%}</sub>',
            x=0.5,
            font=dict(size=16)
        ),
        height=800,
        width=1300,
        template='plotly_white',
        showlegend=True
    )
    
    fig.update_xaxes(title_text='Returns', row=1, col=1)
    fig.update_yaxes(title_text='Density', row=1, col=1)
    fig.update_yaxes(title_text='Returns', row=1, col=2)
    fig.update_xaxes(title_text='Theoretical Quantiles', row=2, col=1)
    fig.update_yaxes(title_text='Sample Quantiles', row=2, col=1)
    fig.update_xaxes(title_text='Date', row=2, col=2)
    fig.update_yaxes(title_text='Returns', row=2, col=2)
    
    fig.write_html('visualizations/html/target_distribution_dashboard.html')
    fig.write_image('visualizations/plots/target_distribution_dashboard.png', width=1300, height=800)
    
    print("✅ Created: target_distribution_dashboard.html")
    return fig

# ============================================
# 5. FEATURE CLUSTERING DENDROGRAM
# ============================================

def create_feature_clustering_dendrogram(df):
    """Create interactive feature clustering dendrogram"""
    
    # Calculate correlation matrix
    corr_matrix = df.drop('target', axis=1).corr()
    
    # Perform hierarchical clustering
    dissimilarity = 1 - abs(corr_matrix)
    Z = hierarchy.linkage(squareform(dissimilarity), method='average')
    
    # Create dendrogram
    fig = go.Figure()
    
    # Get dendrogram data
    from scipy.cluster.hierarchy import dendrogram
    dendro_data = dendrogram(Z, labels=corr_matrix.columns, no_plot=True)
    
    # Create scatter plot for dendrogram lines
    for i in range(len(dendro_data['icoord'])):
        x_coords = dendro_data['icoord'][i]
        y_coords = dendro_data['dcoord'][i]
        
        fig.add_trace(go.Scatter(
            x=x_coords,
            y=y_coords,
            mode='lines',
            line=dict(color=COLORS['info'], width=2),
            showlegend=False,
            hoverinfo='none'
        ))
    
    # Add leaf labels
    for i, label in enumerate(dendro_data['ivl']):
        fig.add_annotation(
            x=dendro_data['icoord'][-1][i] if i < len(dendro_data['icoord'][-1]) else i * 10,
            y=0,
            text=label,
            showarrow=False,
            textangle=-45,
            font=dict(size=10, color=COLORS['dark']),
            xref='x',
            yref='y'
        )
    
    fig.update_layout(
        title=dict(
            text='<b>🌳 Feature Clustering Dendrogram</b><br><sub>Hierarchical clustering of features based on correlation</sub>',
            x=0.5,
            font=dict(size=16)
        ),
        xaxis=dict(title='Features', showticklabels=False),
        yaxis=dict(title='Distance'),
        height=600,
        width=1200,
        template='plotly_white'
    )
    
    fig.write_html('visualizations/html/feature_clustering_dendrogram.html')
    fig.write_image('visualizations/plots/feature_clustering_dendrogram.png', width=1200, height=600)
    
    print("✅ Created: feature_clustering_dendrogram.html")
    return fig

# ============================================
# MAIN EXECUTION
# ============================================

def main():
    """Generate all feature analysis visualizations"""
    
    print("\n📊 Loading data...")
    
    # Load data
    data_path = PROJECT_ROOT / 'data/features/engineered_features.parquet'
    
    if data_path.exists():
        df = pd.read_parquet(data_path)
        print(f"✅ Loaded {df.shape[0]} samples, {df.shape[1]} features")
    else:
        print(f"⚠️ Data not found at {data_path}")
        print("   Generating sample data for demonstration...")
        
        # Generate sample data
        np.random.seed(42)
        n_samples = 1000
        n_features = 30
        
        # Create correlated features
        base = np.random.normal(0, 1, n_samples)
        features = {}
        
        for i in range(n_features):
            correlation = np.random.uniform(-0.8, 0.8)
            features[f'feature_{i}'] = correlation * base + np.random.normal(0, 0.5, n_samples)
        
        df = pd.DataFrame(features)
        # Add target with some pattern
        df['target'] = 0.5 * features['feature_0'] + 0.3 * features['feature_1'] + np.random.normal(0, 0.5, n_samples)
        
        print(f"✅ Generated {df.shape[0]} samples, {df.shape[1]} features")
    
    # Generate visualizations
    print("\n🎨 Generating visualizations...")
    
    print("\n📊 Creating correlation matrix...")
    create_interactive_correlation_matrix(df.drop('target', axis=1))
    
    print("\n🏆 Creating feature importance analysis...")
    create_animated_feature_importance(df)
    
    print("\n🎯 Creating 3D feature space visualization...")
    create_3d_feature_space_pca(df)
    
    print("\n📈 Creating target distribution dashboard...")
    create_target_distribution_dashboard(df)
    
    print("\n🌳 Creating feature clustering dendrogram...")
    create_feature_clustering_dendrogram(df.drop('target', axis=1))
    
    print("\n" + "="*60)
    print("✅ ALL FEATURE ANALYSIS VISUALIZATIONS GENERATED!")
    print("="*60)
    
    print("\n📁 Output Files:")
    print("   HTML Files: visualizations/html/")
    print("   • correlation_matrix_interactive.html")
    print("   • feature_importance_animated.html")
    print("   • 3d_feature_space_pca.html")
    print("   • target_distribution_dashboard.html")
    print("   • feature_clustering_dendrogram.html")
    
    print("\n📁 Static PNG Files: visualizations/plots/")
    print("   • correlation_matrix_interactive.png")
    print("   • feature_importance_animated.png")
    print("   • 3d_feature_space_pca.png")
    print("   • target_distribution_dashboard.png")
    print("   • feature_clustering_dendrogram.png")
    
    print("\n🎯 Interactive Features:")
    print("   • Hover tooltips with correlation values")
    print("   • Zoom and pan on all charts")
    print("   • 3D rotation for PCA visualization")
    print("   • Toggle legend items")
    print("   • Color-coded feature importance")
    print("   • Dendrogram exploration")
    
    print("\n💡 Usage:")
    print("   • Open HTML files in any web browser")
    print("   • Hover over heatmap cells for exact correlations")
    print("   • Rotate 3D plot to explore feature space")
    print("   • Use zoom to focus on important features")
    print("   • Export as PNG using the camera icon")

if __name__ == "__main__":
    main()

🎨 Generating Animated Feature Analysis Dashboard

📊 Loading data...
⚠️ Data not found at c:\Users\nyvra\Downloads\sp500-predictor\notebooks\06_deployment\data\features\engineered_features.parquet
   Generating sample data for demonstration...
✅ Generated 1000 samples, 31 features

🎨 Generating visualizations...

📊 Creating correlation matrix...
